# 🎬 NotebookLM Watermark Remover & Logo Overlay
**A lightweight, dedicated tool to hide the NotebookLM watermark and trim the Gemini outro.**

In [1]:
# Check if a GPU is actually attached to this session
!nvidia-smi -L || echo "⚠️  No GPU detected by nvidia-smi. Go to Settings > Accelerator > GPU (e.g. T4 x2) in this notebook, then re-run."

!pip install -q gradio

# The apt-get ffmpeg build on Kaggle often does NOT include h264_nvenc support.
# Pull a static build that has nvenc compiled in, so GPU encoding actually works.
import os, subprocess
if not os.path.exists('/usr/local/bin/ffmpeg_gpu/ffmpeg'):
    os.makedirs('/usr/local/bin/ffmpeg_gpu', exist_ok=True)
    subprocess.run("wget -q https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz -O /tmp/ffmpeg.tar.xz", shell=True, check=True)
    subprocess.run("tar -xf /tmp/ffmpeg.tar.xz -C /tmp/", shell=True, check=True)
    subprocess.run("cp /tmp/ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /tmp/ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffmpeg_gpu/", shell=True, check=True)

os.environ['PATH'] = '/usr/local/bin/ffmpeg_gpu:' + os.environ['PATH']

print('ffmpeg build in use:')
!which ffmpeg
!ffmpeg -version | head -1
print('nvenc encoders available:')
!ffmpeg -hide_banner -encoders 2>/dev/null | grep nvenc || echo "  none found — will fall back to CPU (libx264) automatically"
print('✅ Dependencies installed!')

GPU 0: Tesla T4 (UUID: GPU-f7845e36-7efb-1d12-828e-3a6fcb8bd333)
GPU 1: Tesla T4 (UUID: GPU-526b6cf7-b0c5-ccc0-5998-cbdbcd552cd9)
ffmpeg build in use:
/usr/local/bin/ffmpeg_gpu/ffmpeg
ffmpeg version N-126549-ga51bb69b09-20260914 Copyright (c) 2000-2026 the FFmpeg developers
nvenc encoders available:
 V....D av1_nvenc            NVIDIA NVENC av1 encoder (codec av1)
 V....D h264_nvenc           NVIDIA NVENC H.264 encoder (codec h264)
 V....D hevc_nvenc           NVIDIA NVENC hevc encoder (codec hevc)
✅ Dependencies installed!


In [2]:
import gradio as gr
import subprocess
import os
import tempfile

def get_video_dims(video_path):
    cmd = f"ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of csv=p=0:s=x '{video_path}'"
    out = subprocess.check_output(cmd, shell=True).decode('utf-8').strip()
    w, h = out.split('x')
    return int(w), int(h)

def build_filter(w, h, zoom_pct, logo_in, logo_width, x_offset, y_offset):
    # Crop a thin margin off the right/bottom (where the watermark sits) then scale
    # back up to the original size — this "zooms in" just enough to push the
    # watermark out of frame, instead of relying on covering it exactly with a logo.
    crop_w = int(w * (1 - zoom_pct / 100))
    crop_h = int(h * (1 - zoom_pct / 100))
    crop_w -= crop_w % 2  # keep even dims for the encoder
    crop_h -= crop_h % 2

    # flags=lanczos gives a sharper upscale than the default bilinear resize, and a light
    # unsharp pass afterwards recovers most of the edge detail lost from the crop+rescale
    # "digital zoom" — resolution stays exactly {w}x{h} either way, this just keeps it crisp.
    if logo_in:
        return (
            f"[0:v]crop={crop_w}:{crop_h}:0:0,scale={w}:{h}:flags=lanczos,"
            f"unsharp=5:5:0.6:5:5:0.0[base];"
            f"[1:v]format=rgba,scale={logo_width}:-1[logo];"
            f"[base][logo]overlay=W-w-{x_offset}:H-h-{y_offset}:format=auto"
        )
    else:
        return f"crop={crop_w}:{crop_h}:0:0,scale={w}:{h}:flags=lanczos,unsharp=5:5:0.6:5:5:0.0"

def preview_position(video_in, logo_in, logo_width, x_offset, y_offset, zoom_pct):
    if not video_in:
        return None, "❌ Upload a video first."
    w, h = get_video_dims(video_in)
    temp_dir = tempfile.mkdtemp()
    preview_path = os.path.join(temp_dir, "preview.png")
    try:
        filter_str = build_filter(w, h, zoom_pct, logo_in, logo_width, x_offset, y_offset)
        if logo_in:
            cmd = f"ffmpeg -ss 1 -i '{video_in}' -i '{logo_in}' -frames:v 1 -filter_complex '{filter_str}' '{preview_path}' -y"
        else:
            cmd = f"ffmpeg -ss 1 -i '{video_in}' -frames:v 1 -vf '{filter_str}' '{preview_path}' -y"
        subprocess.run(cmd, shell=True, check=True, capture_output=True)
        return preview_path, "✅ Preview updated — adjust Zoom/Offset sliders and preview again until the watermark is gone."
    except subprocess.CalledProcessError as e:
        return None, f"❌ Preview error: {e.stderr.decode('utf-8', errors='ignore')[-500:]}"

def process_video(video_in, logo_in, trim_sec, logo_width, x_offset, y_offset, zoom_pct, progress=gr.Progress()):
    if not video_in:
        yield None, "❌ Please upload a video."
        return

    temp_dir = tempfile.mkdtemp()
    out_path = os.path.join(temp_dir, "final_watermark_removed.mp4")

    try:
        progress(0.1, desc="Analyzing video...")
        dur_cmd = f"ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 '{video_in}'"
        duration = float(subprocess.check_output(dur_cmd, shell=True).decode('utf-8').strip())
        new_dur = max(0.0, duration - trim_sec)
        w, h = get_video_dims(video_in)

        progress(0.4, desc="Cropping watermark corner and applying logo...")

        used_gpu = None
        if logo_in or zoom_pct > 0:
            filter_str = build_filter(w, h, zoom_pct, logo_in, logo_width, x_offset, y_offset)
            filter_flag = "-filter_complex" if logo_in else "-vf"
            logo_input = f"-i '{logo_in}' " if logo_in else ""

            # Try GPU (nvenc) first, quality-based (not flat bitrate) so it doesn't look
            # softer than the source. If nvenc isn't actually available on this session
            # (build without nvenc support, or no GPU attached), fall back to CPU (libx264)
            # automatically instead of erroring out.
            cmd_gpu = (
                f"ffmpeg -i '{video_in}' {logo_input}-t {new_dur} {filter_flag} '{filter_str}' "
                f"-c:v h264_nvenc -preset p6 -rc vbr -cq 19 -b:v 0 -maxrate 25M -bufsize 30M "
                f"-c:a copy '{out_path}' -y"
            )
            cmd_cpu = (
                f"ffmpeg -i '{video_in}' {logo_input}-t {new_dur} {filter_flag} '{filter_str}' "
                f"-c:v libx264 -preset fast -crf 19 -c:a copy '{out_path}' -y"
            )

            print(f"Trying GPU (nvenc): {cmd_gpu}")
            gpu_result = subprocess.run(cmd_gpu, shell=True, capture_output=True)
            if gpu_result.returncode == 0:
                used_gpu = True
            else:
                print("⚠️ GPU (nvenc) encode failed, falling back to CPU:")
                print(gpu_result.stderr.decode('utf-8', errors='ignore')[-800:])
                used_gpu = False
                progress(0.6, desc="GPU unavailable, encoding on CPU instead...")
                print(f"Running CPU fallback: {cmd_cpu}")
                subprocess.run(cmd_cpu, shell=True, check=True)
        else:
            # No logo, no zoom -> pure trim, stream copy, zero quality loss, super fast
            cmd = f"ffmpeg -i '{video_in}' -t {new_dur} -c:v copy -c:a copy '{out_path}' -y"
            print(f"Running command: {cmd}")
            subprocess.run(cmd, shell=True, check=True)

        final_working = '/kaggle/working/FINAL_VIDEO_CLEAN.mp4'
        import shutil
        shutil.copy(out_path, final_working)

        engine_note = " (GPU/nvenc)" if used_gpu else (" (CPU/libx264 — GPU not available, see log above)" if used_gpu is False else "")
        yield out_path, f"✅ Video successfully processed!{engine_note} (Also saved to /kaggle/working/)"
    except Exception as e:
        import traceback
        traceback.print_exc()
        yield None, f"❌ Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Base()) as app:
    gr.Markdown('### ✂️ NotebookLM Watermark Hider & Outro Trimmer')
    gr.Markdown('Set **Zoom** to crop the watermark corner out of frame (recommended, most reliable). Optionally add your own logo on top too. Use **Preview Position** to check before running the full video.')

    with gr.Row():
        with gr.Column():
            v_in = gr.Video(label='1. Upload Raw NotebookLM Video')
            logo_in = gr.Image(type='filepath', image_mode='RGBA', label='2. (Optional) Upload Custom Logo — PNG with transparent background')

            trim_slider = gr.Slider(minimum=0.0, maximum=10.0, value=3.0, step=0.1, label='Trim Outro Seconds (Cuts off Gemini ending)')
            zoom_slider = gr.Slider(minimum=0, maximum=20, value=6, step=1, label='Zoom to Crop Watermark Corner (%)')
            width_slider = gr.Slider(minimum=50, maximum=800, value=180, step=10, label='Logo Width in Pixels')
            x_offset_slider = gr.Slider(minimum=-100, maximum=600, value=20, step=5, label='Logo X Offset from Right Edge')
            y_offset_slider = gr.Slider(minimum=-100, maximum=600, value=20, step=5, label='Logo Y Offset from Bottom Edge')

            preview_btn = gr.Button('👁️ Preview Position (fast, single frame)')
            btn = gr.Button('🚀 Generate Clean Video', variant='primary')

        with gr.Column():
            preview_img = gr.Image(label='📍 Position Preview')
            v_out = gr.Video(label='🎬 Final Output Video')
            status = gr.Textbox(label='Status Log')

    preview_btn.click(
        preview_position,
        inputs=[v_in, logo_in, width_slider, x_offset_slider, y_offset_slider, zoom_slider],
        outputs=[preview_img, status]
    )
    btn.click(
        process_video,
        inputs=[v_in, logo_in, trim_slider, width_slider, x_offset_slider, y_offset_slider, zoom_slider],
        outputs=[v_out, status]
    )

print('✅ UI Ready! Launching...')
app.launch(share=True, allowed_paths=['/kaggle/working', '/tmp'])


/tmp/ipykernel_58/511816836.py:116: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base()) as app:


✅ UI Ready! Launching...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://deb3ecda898c651954.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
